# FERRUS CORPUS — corpus_main.ipynb
## Fregate 02 : Incarnation des animations R15 dans un avatar Roblox

```
IN/  ferrus_P*.fbx  +  IN_AVATAR/avatar_r15.blend
         ↓
   inject_animation.py (Blender headless)
         ↓
OUT/ corpus_P*.blend (MASTER) + corpus_P*.glb (PREVIEW)
```

**POUR L'EMPEROR. POUR LA FLOTTE FERRUS.**

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 1 — SETUP
# Monte le Drive, detecte Blender, prepare les chemins
# ═══════════════════════════════════════════════════════════

import os, glob, subprocess, json
from google.colab import drive

# ── 1.1 Monter Google Drive ──────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ── 1.2 CONFIGURER ICI ───────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/FLOTTE-FERRUS'

# ── 1.3 Chemins derives ──────────────────────────────────
CORPUS_ROOT   = os.path.join(DRIVE_ROOT, '02_FERRUS_CORPUS')
IN_DIR        = os.path.join(CORPUS_ROOT, 'IN')
IN_AVATAR_DIR = os.path.join(CORPUS_ROOT, 'IN_AVATAR')
OUT_DIR       = os.path.join(CORPUS_ROOT, 'OUT')
CODEBASE_DIR  = os.path.join(CORPUS_ROOT, 'codebase')
INJECT_SCRIPT = os.path.join(CODEBASE_DIR, 'inject_animation.py')

os.makedirs(OUT_DIR, exist_ok=True)

# ── 1.4 Detecter Blender ─────────────────────────────────
BLENDER_CANDIDATES = [
    '/content/drive/MyDrive/EXODUS_AI_MODELS/blender-4.0.0-linux-x64/blender',
    '/content/drive/MyDrive/EXODUS_AI_MODELS/BLENDER/blender-4.0.0-linux-x64/blender',
    '/usr/local/blender/blender',
    '/content/blender/blender',
]
BLENDER_BIN = next((p for p in BLENDER_CANDIDATES if os.path.isfile(p)), None)

if not BLENDER_BIN:
    print('[SETUP] Blender non trouve dans les emplacements standards.')
    print('[SETUP] Renseigner manuellement : BLENDER_BIN = "/chemin/vers/blender"')
else:
    print(f'[SETUP] Blender detecte : {BLENDER_BIN}')

print(f'[SETUP] DRIVE_ROOT   : {DRIVE_ROOT}')
print(f'[SETUP] CORPUS_ROOT  : {CORPUS_ROOT}')
print(f'[SETUP] IN           : {IN_DIR}')
print(f'[SETUP] IN_AVATAR    : {IN_AVATAR_DIR}')
print(f'[SETUP] OUT          : {OUT_DIR}')
print('[SETUP] OK')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 2 — CONFIG
# Liste les FBX disponibles + valide l'avatar
# ═══════════════════════════════════════════════════════════

# ── 2.1 Lister les FBX ───────────────────────────────────
fbx_files = sorted(glob.glob(os.path.join(IN_DIR, 'ferrus_P*.fbx')))

if not fbx_files:
    print('[CONFIG] ERREUR — Aucun ferrus_P*.fbx dans IN/')
    print(f'[CONFIG] Deposer les FBX dans : {IN_DIR}')
else:
    print(f'[CONFIG] {len(fbx_files)} FBX trouves dans IN/ :')
    for f in fbx_files:
        size_ko = os.path.getsize(f) // 1024
        print(f'         {os.path.basename(f)}  ({size_ko} Ko)')

# ── 2.2 Valider l'avatar ──────────────────────────────────
avatar_candidates = glob.glob(os.path.join(IN_AVATAR_DIR, '*.blend'))

if not avatar_candidates:
    print('[CONFIG] ERREUR — Aucun .blend dans IN_AVATAR/')
    print(f'[CONFIG] Deposer avatar_r15.blend dans : {IN_AVATAR_DIR}')
    AVATAR_BLEND = None
else:
    AVATAR_BLEND = avatar_candidates[0]
    size_ko = os.path.getsize(AVATAR_BLEND) // 1024
    print(f'[CONFIG] Avatar : {os.path.basename(AVATAR_BLEND)}  ({size_ko} Ko)')

# ── 2.3 Valider le script ────────────────────────────────
if not os.path.isfile(INJECT_SCRIPT):
    print(f'[CONFIG] ERREUR — inject_animation.py introuvable : {INJECT_SCRIPT}')
else:
    print(f'[CONFIG] Script OK : inject_animation.py')

# ── 2.4 Bilan ────────────────────────────────────────────
ready = bool(fbx_files and AVATAR_BLEND and os.path.isfile(INJECT_SCRIPT) and BLENDER_BIN)
print()
print('[CONFIG] BILAN ──────────────────────────────────')
print(f'  FBX dispos   : {len(fbx_files)}')
print(f'  Avatar       : {"OK" if AVATAR_BLEND else "MANQUANT"}')
print(f'  Script       : {"OK" if os.path.isfile(INJECT_SCRIPT) else "MANQUANT"}')
print(f'  Blender      : {"OK" if BLENDER_BIN else "MANQUANT"}')
print(f'  GO           : {"OUI" if ready else "NON — corriger les points ci-dessus"}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 3 — INCARNATION
# Boucle sur N acteurs, appelle inject_animation.py
# ═══════════════════════════════════════════════════════════

import datetime

if not ready:
    print('[INCARNATION] Conditions non remplies — executer la cellule CONFIG dabord')
    raise SystemExit

rapport_global = {
    'generated_at': datetime.datetime.now().isoformat(),
    'blender_bin': BLENDER_BIN,
    'avatar_source': os.path.basename(AVATAR_BLEND),
    'total_actors': len(fbx_files),
    'actors': []
}

for fbx_path in fbx_files:
    stem = os.path.splitext(os.path.basename(fbx_path))[0]  # ex: ferrus_P1
    actor_id = stem.replace('ferrus_', '')                   # ex: P1
    out_blend  = os.path.join(OUT_DIR, f'corpus_{actor_id}.blend')
    out_glb    = os.path.join(OUT_DIR, f'corpus_{actor_id}.glb')
    report_tmp = os.path.join(OUT_DIR, f'_tmp_report_{actor_id}.json')

    print(f'[INCARNATION] ─────────────────────────────────')
    print(f'[INCARNATION] Acteur {actor_id} — {os.path.basename(fbx_path)}')

    cmd = [
        BLENDER_BIN, '--background',
        '--python', INJECT_SCRIPT,
        '--',
        '--fbx',           fbx_path,
        '--avatar',        AVATAR_BLEND,
        '--output-blend',  out_blend,
        '--output-glb',    out_glb,
        '--report-json',   report_tmp,
    ]

    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=300)

    # Afficher les logs CORPUS
    corpus_lines = [l for l in proc.stdout.splitlines() if '[CORPUS]' in l]
    for line in corpus_lines:
        print(line)

    if proc.returncode != 0:
        print(f'[INCARNATION] ERREUR Blender (code {proc.returncode})')
        error_lines = [l for l in proc.stderr.splitlines() if 'Error' in l or 'error' in l]
        for line in error_lines[-5:]:
            print(f'  {line}')
        rapport_global['actors'].append({
            'id': actor_id,
            'fbx_source': os.path.basename(fbx_path),
            'status': 'ERREUR',
            'return_code': proc.returncode
        })
        continue

    # Lire le rapport individuel
    actor_report = {'id': actor_id, 'status': 'OK'}
    if os.path.isfile(report_tmp):
        with open(report_tmp) as f:
            actor_data = json.load(f)
        actor_report.update(actor_data)
        os.remove(report_tmp)

    rapport_global['actors'].append(actor_report)
    print(f'[INCARNATION] Acteur {actor_id} OK')

# Ecrire le rapport global
rapport_path = os.path.join(OUT_DIR, 'rapport_corpus.json')
with open(rapport_path, 'w') as f:
    json.dump(rapport_global, f, indent=2)

ok_count  = sum(1 for a in rapport_global['actors'] if a.get('status') == 'OK')
err_count = len(rapport_global['actors']) - ok_count

print()
print('[INCARNATION] ═══════════════════════════════════')
print(f'[INCARNATION] TERMINE — {ok_count}/{len(fbx_files)} acteurs incarnes')
if err_count:
    print(f'[INCARNATION] ATTENTION — {err_count} erreur(s)')
print(f'[INCARNATION] Rapport : {rapport_path}')
print('[INCARNATION] ═══════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 4 — RAPPORT
# Affiche le rapport_corpus.json
# ═══════════════════════════════════════════════════════════

rapport_path = os.path.join(OUT_DIR, 'rapport_corpus.json')

if not os.path.isfile(rapport_path):
    print('[RAPPORT] rapport_corpus.json introuvable — executer INCARNATION dabord')
else:
    with open(rapport_path) as f:
        r = json.load(f)

    print(f'[RAPPORT] Genere le : {r["generated_at"]}')
    print(f'[RAPPORT] Total acteurs : {r["total_actors"]}')
    print()
    print(f'{"ACTEUR":<10} {"BONES":<8} {"FRAMES":<8} {"BLEND (Ko)":<12} {"GLB (Ko)":<10} {"STATUT"}')
    print('─' * 65)
    for a in r['actors']:
        if a.get('status') == 'OK':
            print(
                f'{a["id"]:<10} '
                f'{a.get("bones_avatar", "?"):<8} '
                f'{a.get("frames", "?"):<8} '
                f'{a.get("output_blend_size_ko", "?"):<12} '
                f'{a.get("output_glb_size_ko", "?"):<10} '
                f'{a["status"]}'
            )
        else:
            print(f'{a["id"]:<10} {"─":<8} {"─":<8} {"─":<12} {"─":<10} ERREUR')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 5 — PREVIEW
# Verifie les fichiers OUT/ et fournit les liens viewer
# ═══════════════════════════════════════════════════════════

glb_files   = sorted(glob.glob(os.path.join(OUT_DIR, 'corpus_P*.glb')))
blend_files = sorted(glob.glob(os.path.join(OUT_DIR, 'corpus_P*.blend')))

print('[PREVIEW] Fichiers produits dans OUT/ :')
print()
print(f'  {"FICHIER":<35} {"TAILLE"}')
print('  ' + '─' * 45)

for path in blend_files + glb_files:
    size_ko = os.path.getsize(path) // 1024
    print(f'  {os.path.basename(path):<35} {size_ko} Ko')

print()
print('[PREVIEW] Pour visualiser les GLB en ligne :')
print('  → https://gltf-viewer.donmccurdy.com')
print('  → Glisser-deposer le fichier corpus_P*.glb')
print()
print('[PREVIEW] Pour utiliser dans EXODUS U01 :')
print(f'  Copier corpus_P*.blend dans :')
print(f'  DRIVE_ROOT/01_ANIMATION_ENGINE/IN_MIXAMO_BASE/')
print()
if blend_files:
    print(f'[PREVIEW] {len(blend_files)} acteur(s) prets pour production.')
else:
    print('[PREVIEW] Aucun fichier corpus produit — executer INCARNATION dabord')